In [1]:
import os
import ee
import json
import geemap
import configparser
import geopandas as gpd

In [2]:
BASE_DIR = os.getcwd()
CONFIG = configparser.ConfigParser()
CONFIG.read(os.path.join(BASE_DIR, 'script_config.ini'))

BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))

DATA_RAW = os.path.join(BASE_PATH, 'raw')
DATA_RESULTS = os.path.join(BASE_PATH, '..', 'results')

In [4]:
ee.Authenticate()
ee.Initialize()

Define an area of interest using a shapefile.

In [5]:
shapefile_path = os.path.join(DATA_RAW, 'Tororo_boundary.shp')
gdf = gpd.read_file(shapefile_path)
geojson_file = json.loads(gdf.to_json())
shapefile = ee.FeatureCollection(geojson_file)
roi = shapefile.geometry()

Download Bands 2, 3 and 4 satellite images of Landsat 8 from 2015 to 2025.

In [6]:
collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(roi)
    .filterDate('2015-01-01', '2025-12-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 20))
    .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']))

Scale surface reflectance bands of the satellite images.

In [7]:
def apply_scale(image):
    bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']
    scaled = image.select(bands).multiply(0.0000275).add(-0.2)
    return image.addBands(scaled, None, True).copyProperties(image, image.propertyNames())

collection = collection.map(apply_scale)

Export images

In [8]:
def export_image(image):
    
    date = image.date().format('YYYYMMdd').getInfo()
    
    task = ee.batch.Export.image.toDrive(
        image=image.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5']).clip(roi),
        description=f'L8_B2-B5_{date}',
        folder='Landsat8_Tororo',
        fileNamePrefix=f'L8_B2-B5_{date}',
        region=roi,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    
    task.start()
    print(f'Export started for {date}')

Convert collection to list

In [9]:
image_list = collection.toList(collection.size())
count = collection.size().getInfo()

In [ ]:
for i in range(count):
    image = ee.Image(image_list.get(i))
    export_image(image)